### read xlsx

In [0]:
sdf_temp = spark.read.excel("/Volumes/sandbox/z_eunmi1_ko/v_eunmi1_ko/Reddish_이슈_MAC_추가-1.xlsx")

# 첫 번째 행을 컬럼명으로 추출
header = sdf_temp.limit(1).collect()[0]
columns = [str(cell) for cell in header]

# 첫 번째 행 제외하고 컬럼명 변경
sdf_no_header = sdf_temp.subtract(sdf_temp.limit(1))
df = sdf_no_header.toDF(*columns)

# 맥 값 전처리 
from pyspark.sql import functions as F
df = df.withColumn(
    "mac_addr_2",
    F.concat_ws(
        ":",
        F.substring("MAC_ADDRESS", 1, 2),
        F.substring("MAC_ADDRESS", 3, 2),
        F.substring("MAC_ADDRESS", 5, 2),
        F.substring("MAC_ADDRESS", 7, 2),
        F.substring("MAC_ADDRESS", 9, 2),
        F.substring("MAC_ADDRESS", 11, 2),
    )
)

# 맥 해쉬 처리
tv_salt = dbutils.secrets.get('admin', 'salt')

df = df.withColumn("mac_addr_hashed", 
                   F.when(
                       F.col("mac_addr_2").isNull() | (F.col("mac_addr_2") == ''), 
                       None)\
                    .otherwise(F.sha2(F.concat(F.col("mac_addr_2"), F.lit(tv_salt)), 256)))

df = df.select("IDX", "mac_addr_hashed")
display(df)

### 2. activation_date - min(crt_date) 구하기

In [0]:
df.createOrReplaceTempView("tmp_mac_addr")

In [0]:
df_result_1 = spark.sql(f''' 
    
    select raw.mac_addr
           , min(raw.crt_date) as min_crt_date 
        --    , first(raw.Platform_code) as platform_code 
        --    , max(raw.last_chg_date) as max_last_chg_date
        --    , date_format(max(raw.last_chg_date), 'yyyy-MM') as last_chg_date_ym
    from   aic_data_ods.tlamp.activation_date raw
    inner join tmp_mac_addr tmp on raw.mac_addr = tmp.mac_addr_hashed
    group by mac_addr
''')
display(df_result_1)

### 3. country_code 및 sum(app_usage_duration) 계산 

In [0]:
df_result_2 = spark.sql("""
    WITH pivot_data AS (
        SELECT 
            mac_addr,
            {0}
        FROM (
            SELECT 
                mac_addr, 
                date_ym, 
                app_usage_duration
            FROM aic_data_mart.tv_usage.app_usage_by_user raw 
            inner join tmp_mac_addr tmp on raw.mac_addr = tmp.mac_addr_hashed
        )
        PIVOT (
            SUM(app_usage_duration) 
            FOR date_ym IN ({1})
        )
    ),
    country_data AS (
        SELECT 
            raw.mac_addr,
            concat_ws(',', collect_set(raw.country_code)) as country_code
        FROM aic_data_mart.tv_usage.app_usage_by_user raw 
        inner join tmp_mac_addr tmp on raw.mac_addr = tmp.mac_addr_hashed
        GROUP BY raw.mac_addr
    )
    SELECT 
        p.*,
        c.country_code
    FROM pivot_data p
    LEFT JOIN country_data c ON p.mac_addr = c.mac_addr
""".format(
    ",".join([f"`{row.date_ym}`" for row in spark.sql("SELECT DISTINCT date_ym FROM aic_data_mart.tv_usage.app_usage_by_user ORDER BY date_ym").collect()]),
    ",".join([f"'{row.date_ym}'" for row in spark.sql("SELECT DISTINCT date_ym FROM aic_data_mart.tv_usage.app_usage_by_user ORDER BY date_ym").collect()])
))

display(df_result_2)

### result

In [0]:
df_result = df \
    .join(df_result_1, df["mac_addr_hashed"] == df_result_1["mac_addr"], "left") \
    .join(df_result_2, df["mac_addr_hashed"] == df_result_2["mac_addr"], "left")

display(df_result)

### csv 로 출력 

In [0]:
# KST today date
from datetime import datetime, timezone, timedelta
kst_now = datetime.now(timezone.utc) + timedelta(hours=9)
kst_date = kst_now.strftime("%Y%m%d")

# notebook name
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_name = notebook_path.split("/")[-1]

# write to csv with utf-8 encoding
df_result = df_result.drop('mac_addr_hashed', 'mac_addr', 'mac_addr.1')
df_result.where("p.mac_addr is not null").coalesce(1).write\
    .format('com.databricks.spark.csv')\
    .option('header', 'true')\
    .option('encoding', 'UTF-8')\
    .mode("overwrite")\
    .save(f's3://s3-lge-he-inbound-aic-dev/HEDS/{notebook_name}/{kst_date}')